# Fine-tune Surya OCR 2.5 (Colab)

Full fine-tuning of `datalab-to/surya-ocr-2.5` on pages from our generator (branch `surya`, tasks `full_ocr` and
`block_ocr`): the vision encoder is frozen, the rest (602M weights) is trained with 8-bit AdamW.

1. *Runtime > Change runtime type > T4 GPU*.
2. Put the batch folders in Google Drive: `MyDrive/surya/batches/epoch_001`, `epoch_002`, ... One folder per epoch,
   each made by `python -m pipeline.dataset --n 1000 --out epoch_001 --task full_ocr+block_ocr --seed 1` (new seed
   per epoch). Folders can be added while training runs: the notebook waits for the next one (`WAIT_MINUTES`).
3. *Runtime > Run all*. Progress is saved to Drive; after a disconnect, *Run all* again continues.

In [ ]:
%pip install -q transformers==5.22.1 bitsandbytes==0.50.2

In [ ]:
import difflib, json, os, random, re, time
from pathlib import Path

import bitsandbytes as bnb
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration, TokenizersBackend, get_cosine_schedule_with_warmup

if "google.colab" in str(get_ipython()):
    from google.colab import drive
    drive.mount("/content/drive")

MODEL = "datalab-to/surya-ocr-2.5"
DATA = Path("/content/drive/MyDrive/surya/batches")  # epoch_001/, epoch_002/, ...
OUT = Path("/content/drive/MyDrive/surya/run1")
EPOCHS = 2
LR = 1e-5
GRAD_ACCUM = 8
MAX_TOKENS = 8192      # longer pages are skipped
SAVE_EVERY = 100       # optimizer steps
EVAL_PAGES = 8
EVAL_MAX_TOKENS = 4096  # longest answer generated when measuring
WAIT_MINUTES = 0       # how long to wait for a batch folder that is not there yet
OUT.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device, torch.cuda.get_device_name(0) if device == "cuda" else "")

**Load the model.** Two settings matter, both checked against Surya's own server:

- the tokenizer is loaded as Surya stores it (`TokenizersBackend`). transformers otherwise loads it as a Qwen3.5
  tokenizer, which splits text differently (Arabic becomes byte fragments, `\n` gets another id): with that, the
  model answers block images with nothing;
- images are resized within Surya's limits (at least 1792x28, at most 3072x2048 pixels).

In [ ]:
model = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float32).to(device)  # a T4 has no bfloat16
processor = AutoProcessor.from_pretrained(MODEL, tokenizer=TokenizersBackend.from_pretrained(MODEL))
processor.image_processor.size = {"shortest_edge": 1792 * 28, "longest_edge": 3072 * 2048}

for part in [model.model.visual.patch_embed, model.model.visual.pos_embed, model.model.visual.blocks]:
    part.requires_grad_(False)  # the vision encoder stays as it is; its merger and the language model train
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"{sum(p.numel() for p in trainable) / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M weights train")

**Data.** Each page: the image and its prompt as the user turn, written by the chat template exactly as Surya is
asked, then the answer as the model writes it: the target's own tokens and the end token `<|im_end|>` (id 2).
The answer is not tokenized together with the chat text: Surya's vocabulary has tokens such as `.<` and `><`, which
would join the target's last character with the `<` of `<|im_end|>` and turn the end token into plain text.
Only the answer counts in the loss (`labels` -100 for the prompt), and in it not the words of picture descriptions
(`<img alt="Photo: ..."/><p class="img-desc">...</p>`): ours are one stock sentence per kind of picture, while Surya
describes each picture; the tags around them are learned.

In [ ]:
def batch(epoch):
    folder = DATA / f"epoch_{epoch:03d}"
    deadline = time.time() + WAIT_MINUTES * 60
    while not (folder / "dataset.json").exists():  # the dataset command writes it last
        if time.time() > deadline:
            raise FileNotFoundError(f"{folder} is not there (or not finished)")
        time.sleep(30)
    rows = [json.loads(line) for line in (folder / "manifest.jsonl").read_text(encoding="utf-8").splitlines()]
    return [dict(row, folder=folder) for row in rows if row["task"] in ("full_ocr", "block_ocr")]


DESCRIPTION = re.compile(r'<img alt="Photo: ([^"]*)"/><p class="img-desc">([^<]*)</p>')


def inputs_for(row, with_answer):
    image = Image.open(row["folder"] / row["images"]).convert("RGB")
    prompt = (row["folder"] / row["prompts"]).read_text(encoding="utf-8")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt")
    if with_answer:
        target = (row["folder"] / row["targets"]).read_text(encoding="utf-8")
        answer = processor.tokenizer(target, add_special_tokens=False, return_offsets_mapping=True)
        spans = [m.span(g) for m in DESCRIPTION.finditer(target) for g in (1, 2)]
        ids = answer["input_ids"] + [processor.tokenizer.eos_token_id]
        # a token at least half inside a picture description: not in the loss
        labels = [-100 if 2 * sum(max(0, min(e, b) - max(s, a)) for a, b in spans) >= e - s > 0 else t
                  for t, (s, e) in zip(answer["input_ids"], answer["offset_mapping"])]
        added = torch.tensor([ids])
        inputs["input_ids"] = torch.cat([inputs["input_ids"], added], 1)
        inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones_like(added)], 1)
        inputs["mm_token_type_ids"] = torch.cat([inputs["mm_token_type_ids"], torch.zeros_like(added)], 1)
        # labels for the last prompt position and the answer (the model shifts them by one)
        inputs["labels"] = torch.tensor([[-100] + labels + [processor.tokenizer.eos_token_id]])
    return inputs.to(device)


val = [row for row in batch(1) if row["split"] == "val"][:EVAL_PAGES] or batch(1)[:EVAL_PAGES]  # no val pages: the first ones


def evaluate(name):
    model.eval()
    scores = []
    for row in val:
        inputs = inputs_for(row, with_answer=False)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=EVAL_MAX_TOKENS, do_sample=False, use_cache=True)
        answer = processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        target = (row["folder"] / row["targets"]).read_text(encoding="utf-8")
        no_words = lambda text: DESCRIPTION.sub('<img alt="Photo: "/><p class="img-desc"></p>', text)  # not trained
        scores.append(difflib.SequenceMatcher(None, no_words(answer), no_words(target)).ratio())
    model.train()
    print(f"{name}: answers {sum(scores) / len(scores):.1%} similar to the targets on {len(scores)} pages")
    return scores


if not (OUT / "before.json").exists():
    (OUT / "before.json").write_text(json.dumps(evaluate("before training")))

**Train.** One epoch per batch folder; progress is saved every `SAVE_EVERY` steps and after every epoch.
After every epoch the validation pages are read again and the similarity is saved to `history.json`.

In [ ]:
steps = EPOCHS * -(-sum(row["split"] == "train" for row in batch(1)) // GRAD_ACCUM)
optimizer = bnb.optim.AdamW8bit(trainable, lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, steps // 20), steps)
state = {"epoch": 1, "page": 0, "step": 0}
if (OUT / "state.pt").exists():
    saved = torch.load(OUT / "state.pt", map_location=device, weights_only=False)
    model.load_state_dict(saved["model"], strict=False)
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    state = saved["state"]
    del saved
    print("continuing from", state)


def save():
    trained = {n: p for n, p in model.named_parameters() if p.requires_grad}
    torch.save({"model": trained, "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "state": state}, OUT / "state.tmp")
    os.replace(OUT / "state.tmp", OUT / "state.pt")


model.train()
while state["epoch"] <= EPOCHS:
    rows = [row for row in batch(state["epoch"]) if row["split"] == "train"]
    random.Random(state["epoch"]).shuffle(rows)
    start, first, losses = time.time(), state["page"], []
    for i in range(first, len(rows)):
        inputs = inputs_for(rows[i], with_answer=True)
        if inputs["input_ids"].shape[1] <= MAX_TOKENS:
            loss = model(**inputs, logits_to_keep=inputs["labels"].shape[1]).loss  # logits only where labels are
            (loss / GRAD_ACCUM).backward()
            losses.append(loss.item())
        if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(rows):
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            state.update(page=i + 1, step=state["step"] + 1)
            gpu = f", GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if device == "cuda" else ""
            print(f"epoch {state['epoch']} step {state['step']}/{steps}: loss {sum(losses) / max(1, len(losses)):.4f}, "
                  f"{(time.time() - start) / (i + 1 - first):.1f} s/page{gpu}")
            losses = []
            if state["step"] % SAVE_EVERY == 0:
                save()
    state.update(epoch=state["epoch"] + 1, page=0)
    save()
    scores = evaluate(f"after epoch {state['epoch'] - 1}")  # the same validation pages every time
    history = json.loads((OUT / "history.json").read_text()) if (OUT / "history.json").exists() else {}
    history[state["epoch"] - 1] = sum(scores) / len(scores)
    (OUT / "history.json").write_text(json.dumps(history, indent=1))

In [ ]:
before = json.loads((OUT / "before.json").read_text())
history = json.loads((OUT / "history.json").read_text())
print(f"before training: {sum(before) / len(before):.1%}")
for epoch, score in history.items():
    print(f"after epoch {epoch}: {score:.1%}")
model.save_pretrained(OUT / "model")
processor.save_pretrained(OUT / "model")
print("model saved to", OUT / "model")